In [9]:
#from langchain_google_vertexai import ChatVertexAI
import google.generativeai as genai
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from pydantic import BaseModel, Field
from config import GEMINI_KEY
import pandas as pd
import os
pd.set_option('display.max_colwidth', None)

genai.configure(api_key= GEMINI_KEY)

In [ ]:
item = pd.read_csv('../../data/preprocessed_item.csv')
youtube = pd.read_csv('../../data/youtube_data.csv', lineterminator='\n')

In [11]:
youtube['input'] = youtube['title'] + ' ' + youtube['video_description']

In [12]:
from typing import List

llm = ChatGoogleGenerativeAI(model = "gemini-1.5-flash-8b", temperature=0)

# Pydantic
class Extract(BaseModel):
    is_cosmetic: bool = Field(description="The given text is related to cosmetics")
    cosmetic_list: List[str] = Field(description="The list of the cosmetics names")

structured_llm = llm.with_structured_output(Extract)

In [13]:
examples = [
    HumanMessage("#shorts #올리브영아이라이너 #지속력좋은아이라이너 #클리오샤프쏘심플워터프루프펜슬라이너 #코스노리슈퍼프루프피팅젤아이라이너 #머지더퍼스트펜아이라이너 #클리오워터프루프펜라이너킬브라운 #웨이크메이크철벽펜아이라이너", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "is_cosmetic": True,
                    "cosmetic_list": ["클리오샤프쏘심플워터프루프펜슬라이너", "코스노리슈퍼프루프피팅젤아이라이너", "머지더퍼스트펜아이라이너", "클리오워터프루프펜라이너킬브라운", "웨이크메이크철벽펜아이라이너"]},
                "id": "1",
            }
        ],
    ),
    ToolMessage("", tool_call_id="1"),
    HumanMessage("💛 한율 달빛유자 패드 X 올리브영 프로모션 (~11/29) 💛🔎 5분 잡티톤업 에센스 패드: https://bit.ly/3ZkzK8E 정가 29,000원 → 23,200원 (20% OFF) #한율 #달빛유자패드 #5분에센스패드 #비타민패드", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "is_cosmetic": True,
                    "cosmetic_list": ["한율 달빛유자 패드", "5분 잡티톤업 에센스 패드"]},
                "id": "2",
            }
        ],
    ),
    ToolMessage("", tool_call_id="2"),
    HumanMessage("👉🏻SKINFOOD : 스킨푸드 00:52 연어 다크서클 컨실러 3colors / 8,000원 직접구매 🔗https://www.oliveyoung.co.kr/store/go...  👉🏻Dr.Jart+ : 닥터자르트 02:28 시카페어 인텐시브 수딩리페어 크림 50ml / 50,000원 직접구매 🔗https://www.oliveyoung.co.kr/store/go... " + \
                "👉🏻MAMONDE : 마몽드 04:35 플로라 글로우 로즈 리퀴드 마스크 80ml / 30,000원 직접구매 (직접구매해서 쭉 사용하다가 광고도 받았던 제품!) 🔗https://www.oliveyoung.co.kr/store/go...", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "is_cosmetic": True,
                    "cosmetic_list": ["연어 다크서클 컨실러", "시카페어 인텐시브 수딩리페어 크림", "플로라 글로우 로즈 리퀴드 마스크"]},
                "id": "3",
            }
        ],
    ),
    ToolMessage("", tool_call_id="3"),
    HumanMessage("[Yes to Myeslf 나를 위한 용기- 부족해서 아름다운 나에게] 도서   교보문고 https://product.kyobobook.co.kr/detai... yes24 https://www.yes24.com/Product/Goods/1... 알라딘 http://aladin.kr/p/6q5Jr  [코어 마인드] 도서 나답게 살기 위해 필요한 것은 강한 내면의 힘 https://product.kyobobook.co.kr/detai...  [코어 마인드] 강의 내 마음대로 살 수 있는 내면의 힘 https://me2.do/FiETCEum  [세상에서 가장 쉬운 본질 육아] 도서  https://bit.ly/3UoHNNa 인세의 일부는 사단법인 야나 후원에 쓰입니다. ", 
                name="user"),
    AIMessage(
        "",
        name="extract_assisstant",
        tool_calls=[
            {
                "name": "extract",
                "args": {
                    "is_cosmetic": True,
                    "cosmetic_list": []},
                "id": "4",
            }
        ],
    ),
    ToolMessage("", tool_call_id="4"),
]
# 시스템 프롬프트
system = """주어진 텍스트가 화장품에 관련된 글인지 판별하고 언급하고 있는 화장품 이름을 모두 찾아주세요. 
    반환 형식은 JSON으로, "is_cosmetic"과 "cosmetic_list" 키를 사용해 반환합니다."""

# ChatPromptTemplate 생성
prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("placeholder", "{examples}"), ("human", "{input}")]
)

# 모델과 프롬프트 연결
few_shot_structured_llm = prompt | structured_llm

def invoke(input):
    result = few_shot_structured_llm.invoke({"input": input, "examples": examples})
    return result.is_cosmetic, result.cosmetic_list

In [14]:
from concurrent.futures import ThreadPoolExecutor
from tenacity import (retry,stop_never,wait_random_exponential)

@retry(wait=wait_random_exponential(min=1, max=20), stop=10)
def process_row(input):
    return invoke(input)

with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_row, youtube['input']))

youtube['is_cosmetic'] = [r[0] for r in results]
youtube['product'] = [r[1] for r in results]

In [9]:
# results = youtube['input'].apply(invoke)
# youtube['is_cosmetic'] = [r[0] for r in results]
# youtube['product'] = [r[1] for r in results]

In [ ]:
# 라벨링한 데이터에 한해 Accuracy 확인
result = youtube[250:]

def accurate(row):
    ans = row['평가 결과']
    #pred = row['label']
    pred = row['is_cosmetic']
    if ans == "1" and pred:
        return 1
    elif ans == "0" and not pred:
        return 1
    else:
        return 0

result['result'] = result.apply(accurate, axis=1)
print(sum(result['result']/len(result)))

In [7]:
#youtube.to_csv('./data/youtube_label.csv', index=False)